In [ ]:
# EXP (no es del baseline): Kaggle monta los inputs en otra carpeta cuando el notebook se crea por API.
# Arma /tmp/lab/input/<nombre> -> carpeta real y se para en /tmp/lab/work, asi '../input/<nombre>/...' del baseline funciona sin tocarlo.
import os, glob
def _buscar(nombre, archivo):
    for prof in range(0, 5):
        patron = "/kaggle/input/" + "*/" * prof
        for d in glob.glob(patron):
            d = d.rstrip("/")
            if os.path.basename(d).endswith(nombre) and os.path.exists(os.path.join(d, archivo)):
                return d
    raise FileNotFoundError(f"{nombre}/{archivo} no aparece en /kaggle/input: {sorted(glob.glob('/kaggle/input/*/*'))[:50]}")
_inputs = {"petfinder-adoption-prediction": "train/train.csv",
           "examen-lab-ii-preprocesado-texto": "train.parquet",
           "examen-lab-ii-preprocesado-img": "train.parquet"}
if not all(os.path.exists(f"../input/{n}/{a}") for n, a in _inputs.items()):
    os.makedirs("/tmp/lab/input", exist_ok=True)
    os.makedirs("/tmp/lab/work", exist_ok=True)
    for n, a in _inputs.items():
        destino = f"/tmp/lab/input/{n}"
        if not os.path.exists(destino):
            os.symlink(_buscar(n, a), destino)
    os.chdir("/tmp/lab/work")
print({n: os.path.realpath(f"../input/{n}") for n in _inputs})

In [ ]:
# Importación de Librerías
import pandas as pd 

from sklearn.model_selection import KFold
from sklearn.metrics import cohen_kappa_score
from lightgbm import LGBMClassifier, early_stopping

def metric(y_true, y_pred):
    res = cohen_kappa_score(y_true, y_pred.reshape((y_true.shape[0], 5), order="F").argmax(axis=1), weights= 'quadratic')
    return "kappa", res, True

In [ ]:
train = pd.read_csv('../input/petfinder-adoption-prediction/train/train.csv').set_index("PetID")
test = pd.read_csv('../input/petfinder-adoption-prediction/test/test.csv').set_index("PetID")
folds = [train.index[test_idx] for train_idx, test_idx in KFold(5).split(train)]

In [ ]:
for c in ['Type', 'Breed1', 'Breed2', 'Gender', 'Color1', 'Color2',
       'Color3', 'FurLength', 'MaturitySize', 'Vaccinated', 'Dewormed',
       'Sterilized', 'Health', 'State', 'RescuerID']:
    train[c] = train[c].astype("category")    
    test[c] = test[c].astype("category")

In [ ]:
train = train\
    .join(pd.read_parquet("../input/examen-lab-ii-preprocesado-texto/train.parquet"))\
    .join(pd.read_parquet("../input/examen-lab-ii-preprocesado-img/train.parquet"))\
    .select_dtypes(exclude="O")
train

In [ ]:
test = test\
    .join(pd.read_parquet("../input/examen-lab-ii-preprocesado-texto/test.parquet"))\
    .join(pd.read_parquet("../input/examen-lab-ii-preprocesado-img/test.parquet"))\
    .select_dtypes(exclude="O")
test

In [ ]:
fold_preds = []
test_preds = []
for i, fold in enumerate(folds):
    X_train = train.drop(fold).drop("AdoptionSpeed", axis=1).select_dtypes(exclude="O")
    y_train = train.drop(fold)["AdoptionSpeed"]

    X_valid = train.loc[fold].drop("AdoptionSpeed", axis=1).select_dtypes(exclude="O")
    y_valid = train.loc[fold, "AdoptionSpeed"]

    learner = LGBMClassifier()
    learner.fit(X_train, y_train, )
    
    learner.fit(X_train, y_train, eval_set=[(X_train, y_train), (X_valid, y_valid)],
                eval_metric= metric, callbacks=[early_stopping(5)])
    fold_preds.append(pd.Series(learner.predict(X_valid), index=fold, name="AdoptionSpeed"))
    test_preds.append(pd.DataFrame(learner.predict_proba(test[X_train.columns]), index=test.index))

fold_preds = pd.concat(fold_preds)
test_preds = sum(test_preds).idxmax(axis=1).rename("AdoptionSpeed")

In [ ]:
# EXP (no es del baseline): una funcion que corre el loop del baseline con variantes.
# svd_img=True: reduce img_feat_* con TruncatedSVD(20) ajustado en el train del fold (EXP-008).
# modelo="reg": LGBMRegressor + cortes por cuantiles de la distribucion de clases del train del fold (EXP-009).
# modelo="clf": LGBMClassifier, igual que el baseline.
import numpy as np
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import KFold, StratifiedGroupKFold
from lightgbm import LGBMRegressor

pred_base = fold_preds
N_SVD = 20
IMG = [c for c in train.columns if c.startswith("img_feat_")]

def reducir(X, svd):
    Z = pd.DataFrame(np.nan, index=X.index, columns=[f"img_svd_{i}" for i in range(N_SVD)])
    tiene = X[IMG].notna().all(axis=1)
    if tiene.any():
        Z.loc[tiene] = svd.transform(X.loc[tiene, IMG].values)
    return X.drop(columns=IMG).join(Z)

def metric_reg(y_true, y_pred):
    return "kappa", cohen_kappa_score(y_true, np.clip(np.rint(y_pred), 0, 4).astype(int), weights="quadratic"), True

def cortes_cuantiles(p, y):
    props = y.value_counts(normalize=True).sort_index().reindex(range(5), fill_value=0).values
    return np.quantile(p, np.cumsum(props)[:-1])

def correr_variante(train, test, folds, svd_img, modelo):
    fold_preds = []
    test_out = []
    for i, fold in enumerate(folds):
        X_train = train.drop(fold).drop("AdoptionSpeed", axis=1).select_dtypes(exclude="O")
        y_train = train.drop(fold)["AdoptionSpeed"]

        X_valid = train.loc[fold].drop("AdoptionSpeed", axis=1).select_dtypes(exclude="O")
        y_valid = train.loc[fold, "AdoptionSpeed"]

        X_test = test[X_train.columns]
        if svd_img:
            svd = TruncatedSVD(n_components=N_SVD, random_state=42)
            svd.fit(X_train.loc[X_train[IMG].notna().all(axis=1), IMG].values)
            X_test = reducir(X_test, svd)
            X_train, X_valid = reducir(X_train, svd), reducir(X_valid, svd)

        if modelo == "clf":
            learner = LGBMClassifier()
            learner.fit(X_train, y_train, )
            learner.fit(X_train, y_train, eval_set=[(X_train, y_train), (X_valid, y_valid)],
                        eval_metric= metric, callbacks=[early_stopping(5)])
            fold_preds.append(pd.Series(learner.predict(X_valid), index=fold, name="AdoptionSpeed"))
            test_out.append(pd.DataFrame(learner.predict_proba(X_test), index=test.index))
        else:
            learner = LGBMRegressor()
            learner.fit(X_train, y_train, )
            learner.fit(X_train, y_train, eval_set=[(X_train, y_train), (X_valid, y_valid)],
                        eval_metric= metric_reg, callbacks=[early_stopping(5)])
            p_valid = learner.predict(X_valid)
            fold_preds.append(pd.Series(np.digitize(p_valid, cortes_cuantiles(p_valid, y_train)), index=fold, name="AdoptionSpeed"))
            test_out.append(pd.Series(learner.predict(X_test), index=test.index))

    fold_preds = pd.concat(fold_preds)
    if modelo == "clf":
        t = sum(test_out).idxmax(axis=1)
    else:
        p_test = sum(test_out) / len(test_out)
        t = pd.Series(np.digitize(p_test, cortes_cuantiles(p_test, train["AdoptionSpeed"])), index=test.index)
    return fold_preds, t.rename("AdoptionSpeed")

def qwk_por_fold(pred, folds):
    return [cohen_kappa_score(train.loc[f, "AdoptionSpeed"], pred.loc[f], weights="quadratic") for f in folds]


In [ ]:
# EXP-017: SVD de imagen (EXP-008) + regresor con cortes por cuantiles (EXP-009), contra el baseline clasificador
pred_exp, test_preds = correr_variante(train, test, folds, True, "reg")


In [ ]:
res = pd.DataFrame({"baseline": qwk_por_fold(pred_base, folds), "exp_017": qwk_por_fold(pred_exp, folds)},
                   index=[f"fold_{i}" for i in range(len(folds))])
res["delta"] = res["exp_017"] - res["baseline"]
print(res.round(4).to_string())
print("media baseline", round(res["baseline"].mean(), 4), "desvio", round(res["baseline"].std(), 4))
print("media exp_017 ", round(res["exp_017"].mean(), 4), "desvio", round(res["exp_017"].std(), 4))
print("delta media   ", round(res["delta"].mean(), 4))
print("folds que mejoran:", int((res["delta"] > 0).sum()), "de", len(folds))


In [ ]:
test_preds.to_csv("submission.csv")
!head submission.csv

In [ ]:
# EXP (no es del baseline): si se cambio de carpeta, dejar submission.csv en el output de Kaggle
import shutil
if os.getcwd() != "/kaggle/working" and os.path.exists("submission.csv"):
    shutil.copy("submission.csv", "/kaggle/working/submission.csv")